# 06 Test Set Evaluation

This notebook evaluates one trained run on the held-out test split using deterministic masking.

## Inputs

- `best_model/` for the selected experiment
- `test_dataset.pt` for the matching tokenizer family and setting label
- `run_index.csv` in the project registry folder

## Outputs

- saved summary metrics
- per-class metrics
- top-1 correctness ROC and precision-recall plots plus summary tables
- qualitative probe results
- an updated run-index entry


## User settings

Update these values before running the notebook.

**What this cell does**
- sets the project location in Google Drive
- selects the trained run to evaluate
- controls masking, batch size, and overwrite behavior

**Expected output**
- a short summary of the selected project root and run settings

**How to interpret**
- if the printed tokenizer family or experiment name is wrong, fix it here before continuing


In [ ]:
from pathlib import Path

# Update PROJECT_ROOT if the project folder lives somewhere else in Google Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# Change these only if the notebook should sync from a different repository or branch.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'

# Map each tokenizer family to the saved setting label used for its tokenized test set.
TOKENIZER_SETTINGS = {
    'byte_bpe': 'v300_m2',
    'glyberta': 'v1_train_only',
    'manual': 'v1_train_only',
    'hybrid_char_bpe': 'v70_m2',
    'linkage_block': 'v1_train_only',
    'donor_bound': 'v1_train_only',
    'semi_atomic': 'v1_train_only',
}

# Choose the trained run to evaluate. The tokenizer family must match both the
# checkpoint folder and the saved test dataset.
TOKENIZER_FAMILY = 'byte_bpe'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv300_m2_cont_lr5e-05_ep20'

# These settings control masking and batch size during evaluation.
MLM_PROBABILITY = 0.15
MASK_SEED = 42
BATCH_SIZE = 16

# If False, the notebook stops instead of replacing existing output files.
OVERWRITE_EXISTING_OUTPUTS = False

print(f'Project root: {PROJECT_ROOT}')
print(f'Tokenizer family: {TOKENIZER_FAMILY}')
print(f'Experiment name: {EXPERIMENT_NAME}')
print(f'Overwrite existing outputs: {OVERWRITE_EXISTING_OUTPUTS}')


## Runtime setup

This cell prepares the Colab runtime for the notebook.

**What this cell does**
- mounts Google Drive
- clones or updates the GitHub repository
- adds the repository root to the Python import path

**Expected output**
- confirmation that Drive is mounted
- confirmation that the repository was found or cloned
- confirmation of the local repository directory

**How to interpret**
- if repository sync fails here, stop and fix that before running the evaluation cells


In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# Mount Google Drive so the notebook can read checkpoints, datasets, and saved results.
drive.mount('/content/drive')

repo_url = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
repo_dir = Path('/content') / REPO_NAME

# Clone the repository in a fresh Colab session, or update the existing clone.
if not repo_dir.exists():
    print(f'Cloning repository from {repo_url} ...')
    subprocess.run(['git', 'clone', '--quiet', repo_url, str(repo_dir)], check=True)
else:
    print(f'Repository already exists at {repo_dir}.')

print(f"Updating repository to the latest '{GITHUB_REF}' changes...")
subprocess.run(
    ['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', GITHUB_REF],
    check=True,
)

# Add the repository root so notebook cells can import shared code from src/.
repo_dir_str = str(repo_dir)
if repo_dir_str not in sys.path:
    sys.path.insert(0, repo_dir_str)

print(f'Repository directory: {repo_dir}')


## Import shared helpers

This cell imports the libraries and shared helper functions used throughout the notebook.

**What this cell does**
- imports analysis libraries such as `pandas` and `torch`
- imports the helper functions that handle test evaluation steps

**Expected output**
- no printed output if imports succeed

**How to interpret**
- an import error usually means the repository did not sync correctly or the branch is missing expected code


In [ ]:
import json

import pandas as pd
import torch
from IPython.display import display

from src.run_index import upsert_run_record
from src.test_evaluation import (
    build_masked_test_dataset,
    build_masking_summary,
    build_test_evaluation_paths,
    build_test_summary_row,
    compute_per_class_metrics,
    compute_summary_classification_metrics,
    compute_topk_metrics,
    get_core_probe_cases,
    load_test_artifacts,
    plot_top1_correctness_pr_curves,
    plot_top1_correctness_roc_curves,
    resolve_setting_label,
    run_mlm_test_predictions,
    run_structured_qualitative_probe,
    validate_test_evaluation_run,
)


## Build paths and validate the run

This cell builds the input and output paths for the selected run and checks that the run is ready to evaluate.

**What this cell does**
- resolves the saved dataset setting label for the tokenizer family
- builds the model, dataset, results, and run-index paths
- checks that required inputs exist
- checks whether outputs already exist when overwriting is disabled

**Expected output**
- a short table showing the main paths for this run
- a confirmation that validation checks passed

**How to interpret**
- a file error usually means the selected run or project root needs to be corrected


In [ ]:
# Look up the saved dataset label for the selected tokenizer family.
SETTING_LABEL = resolve_setting_label(TOKENIZER_FAMILY, TOKENIZER_SETTINGS)

# Build the standard input and output paths used by this notebook.
paths = build_test_evaluation_paths(
    project_root=PROJECT_ROOT,
    tokenizer_family=TOKENIZER_FAMILY,
    setting_label=SETTING_LABEL,
    experiment_name=EXPERIMENT_NAME,
)

# Stop here if the model, test dataset, or overwrite policy is not valid.
validate_test_evaluation_run(
    paths=paths,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
)

path_summary = pd.DataFrame(
    {
        'item': [
            'setting_label',
            'best_model_dir',
            'test_dataset_path',
            'results_dir',
            'run_index_path',
        ],
        'value': [
            SETTING_LABEL,
            str(paths.best_model_dir),
            str(paths.test_dataset_path),
            str(paths.results_dir),
            str(paths.run_index_path),
        ],
    }
)

display(path_summary)
print('Input artifact and output overwrite checks passed.')


## Load the trained model and test dataset

This cell loads the saved best model, tokenizer, and matching tokenized test split.

**What this cell does**
- chooses the compute device
- loads the tokenizer and best model checkpoint
- loads the tokenized test dataset for the selected run

**Expected output**
- the active device
- the number of test sequences loaded

**How to interpret**
- the sequence count should look reasonable for the test split you expect to evaluate


In [ ]:
# Use a GPU when one is available because evaluation can be slow on the CPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load the tokenizer, trained model, and tokenized test dataset for this run.
tokenizer, model, test_dataset = load_test_artifacts(
    model_dir=paths.best_model_dir,
    test_dataset_path=paths.test_dataset_path,
    device=device,
)

print(f'Loaded tokenizer and model from: {paths.best_model_dir}')
print(f'Loaded test dataset with {len(test_dataset)} sequences')


## Build a deterministic masked test set

This cell creates a reproducible masked version of the held-out test split.

**What this cell does**
- applies masking using the chosen probability and seed
- builds a small summary of the masking setup
- saves that summary for later reference

**Expected output**
- a masking summary table
- confirmation that the summary file was saved

**How to interpret**
- rerunning with the same seed should produce the same masking summary values


In [ ]:
# Build the deterministic masked version of the held-out test split.
masked_dataset_dict = build_masked_test_dataset(
    test_dataset=test_dataset,
    tokenizer=tokenizer,
    mlm_probability=MLM_PROBABILITY,
    seed=MASK_SEED,
)

# Save a short record of how the test set was masked for this run.
masking_summary = build_masking_summary(
    masked_dataset_dict=masked_dataset_dict,
    mlm_probability=MLM_PROBABILITY,
    mask_seed=MASK_SEED,
)

display(masking_summary)
masking_summary.to_csv(paths.masking_summary_path, index=False)

print(f'Masking summary saved to: {paths.masking_summary_path}')


## Run masked-token predictions

This cell runs the model on the masked test set.

**What this cell does**
- scores every masked position in the test set
- stores the outputs needed for later metrics
- tracks sequence-level top-1 and top-3 correctness flags

**Expected output**
- a short table showing how many masked tokens and sequences were evaluated

**How to interpret**
- `masked_token_predictions` counts prediction opportunities, while `evaluated_sequences` counts sequences that contributed at least one masked position


In [ ]:
# Run batched predictions and keep both token-level outputs and
# sequence-level correctness flags for later summaries.
y_true, y_pred, y_probs, sequence_top1_flags, sequence_top3_flags = run_mlm_test_predictions(
    model=model,
    masked_dataset_dict=masked_dataset_dict,
    batch_size=BATCH_SIZE,
    device=device,
)

prediction_summary = pd.DataFrame(
    {
        'metric': ['masked_token_predictions', 'evaluated_sequences'],
        'value': [len(y_true), len(sequence_top1_flags)],
    }
)

display(prediction_summary)


## Compute and save summary metrics

This cell computes the main evaluation metrics for the run.

**What this cell does**
- computes top-k recovery metrics
- computes macro and weighted classification metrics
- saves the results in JSON and one-row CSV form

**Expected output**
- a summary metrics table
- confirmation that the JSON and CSV files were saved

**How to interpret**
- compare macro and weighted metrics to see whether performance is balanced across common and rare token classes


In [ ]:
# Compute token-level and sequence-level top-k recovery metrics.
topk_metrics = compute_topk_metrics(
    y_true=y_true,
    y_probs=y_probs,
    y_pred=y_pred,
    sequence_top1_flags=sequence_top1_flags,
    sequence_top3_flags=sequence_top3_flags,
)

# Compute macro and weighted classification metrics across masked token classes.
classification_metrics = compute_summary_classification_metrics(
    y_true=y_true,
    y_pred=y_pred,
)

all_metrics = {**topk_metrics, **classification_metrics}
summary_df = pd.DataFrame(
    {
        'metric': list(all_metrics.keys()),
        'value': list(all_metrics.values()),
    }
)

display(summary_df)

# Save the full metric dictionary and a one-row comparison table.
paths.test_summary_json_path.write_text(
    json.dumps(all_metrics, indent=2),
    encoding='utf-8',
)

test_summary_row = build_test_summary_row(
    tokenizer_family=TOKENIZER_FAMILY,
    setting_label=SETTING_LABEL,
    experiment_name=EXPERIMENT_NAME,
    metrics_dict=all_metrics,
)
test_summary_row.to_csv(paths.test_summary_row_path, index=False)

print(f'Test summary JSON saved to: {paths.test_summary_json_path}')
print(f'Test summary row saved to: {paths.test_summary_row_path}')


## Review per-class metrics

This cell shows how performance changes across token classes.

**What this cell does**
- computes per-class precision, recall, F1, and support
- shows the most common classes first
- saves the full table to CSV

**Expected output**
- the top rows of the per-class metrics table
- confirmation that the CSV file was saved

**How to interpret**
- `support` is how often that token was the correct masked target in the test set


In [ ]:
# Break the overall results out by true token class.
per_class_metrics = compute_per_class_metrics(
    y_true=y_true,
    y_pred=y_pred,
    tokenizer=tokenizer,
)

display(per_class_metrics.head(20))
per_class_metrics.to_csv(paths.per_class_metrics_path, index=False)

print(f'Per-class metrics saved to: {paths.per_class_metrics_path}')


## Plot top-1 correctness ROC curves

This cell plots ROC curves for top-1 correctness.

**What this cell does**
- builds per-class ROC curves using correct versus incorrect top-1 predictions
- aggregates them into macro and weighted summaries
- saves the plot and summary tables

**Expected output**
- an ROC plot
- ROC summary tables

**How to interpret**
- higher AUC values mean top-1 confidence is doing a better job separating correct predictions from incorrect ones


In [ ]:
# Plot macro and support-weighted ROC curves for top-1 correctness.
top1_roc_summary, top1_roc_per_class = plot_top1_correctness_roc_curves(
    y_true=y_true,
    y_pred=y_pred,
    y_probs=y_probs,
    tokenizer=tokenizer,
    save_path=paths.top1_roc_plot_path,
)

display(top1_roc_summary)
display(top1_roc_per_class.head(20))

top1_roc_summary.to_csv(paths.top1_roc_summary_path, index=False)
top1_roc_per_class.to_csv(paths.top1_roc_per_class_path, index=False)

print(f'Top-1 correctness ROC plot saved to: {paths.top1_roc_plot_path}')
print(f'Top-1 correctness ROC summary saved to: {paths.top1_roc_summary_path}')
print(f'Top-1 correctness per-class summary saved to: {paths.top1_roc_per_class_path}')


## Plot top-1 correctness precision-recall curves

This cell plots the weighted precision-recall curve for the same top-1 correctness question.

**What this cell does**
- builds per-class precision-recall curves
- combines them into a weighted summary curve
- saves the plot and summary tables

**Expected output**
- a precision-recall plot
- precision-recall summary tables

**How to interpret**
- this view can be easier to read than ROC when correct predictions are relatively sparse within a class


In [ ]:
# Plot the weighted precision-recall curve for top-1 correctness.
top1_pr_summary, top1_pr_per_class = plot_top1_correctness_pr_curves(
    y_true=y_true,
    y_pred=y_pred,
    y_probs=y_probs,
    tokenizer=tokenizer,
    save_path=paths.top1_pr_plot_path,
)

display(top1_pr_summary)
display(top1_pr_per_class.head(20))

top1_pr_summary.to_csv(paths.top1_pr_summary_path, index=False)
top1_pr_per_class.to_csv(paths.top1_pr_per_class_path, index=False)

print(f'Top-1 correctness PR plot saved to: {paths.top1_pr_plot_path}')
print(f'Top-1 correctness PR summary saved to: {paths.top1_pr_summary_path}')
print(f'Top-1 correctness per-class summary saved to: {paths.top1_pr_per_class_path}')


## Run qualitative biological probes

This cell runs a small set of hand-picked masked examples.

**What this cell does**
- loads shared probe cases
- runs tokenizer-specific fill-mask predictions
- saves the probe results for later review

**Expected output**
- a table of probe predictions
- confirmation that the probe results were saved

**How to interpret**
- `is_expected = True` means the predicted token matches the intended target for that probe case


In [ ]:
# Load the shared probe cases and run tokenizer-specific fill-mask predictions.
probe_cases = get_core_probe_cases()
qualitative_probe_results = run_structured_qualitative_probe(
    model=model,
    tokenizer=tokenizer,
    tokenizer_family=TOKENIZER_FAMILY,
    probe_cases=probe_cases,
    device=device,
)

display(qualitative_probe_results)
qualitative_probe_results.to_csv(paths.qualitative_probe_path, index=False)

print(f'Qualitative probe results saved to: {paths.qualitative_probe_path}')


## Update the run index

This cell records where the main evaluation outputs were saved.

**What this cell does**
- inserts or updates the matching run-index row
- stores the main output paths for this evaluated run
- displays the updated row for a quick check

**Expected output**
- the updated run-index row for this experiment
- confirmation that `run_index.csv` was updated

**How to interpret**
- confirm that the experiment name, tokenizer family, and saved paths match the run you just evaluated


In [ ]:
# Record the main output paths for this evaluated run.
updated_run_index = upsert_run_record(
    paths.run_index_path,
    {
        'experiment_name': EXPERIMENT_NAME,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'results_dir': str(paths.results_dir),
        'test_metrics_path': str(paths.test_summary_json_path),
        'qualitative_probe_path': str(paths.qualitative_probe_path),
        'notebook_used': 'notebooks/06_test_set_evaluation.ipynb',
        'run_status': 'tested',
        'notes': '',
    },
)

run_record_mask = (
    (updated_run_index['experiment_name'] == EXPERIMENT_NAME)
    & (updated_run_index['tokenizer_family'] == TOKENIZER_FAMILY)
)

display(updated_run_index.loc[run_record_mask])
print(f'Run index updated: {paths.run_index_path}')


In [ ]:
# Show the top three masked-token candidates for the branched Gal example
# from the presentation figure.
example_probe_by_family = {
    'manual': {
        'base_sequence': 'Fuca1-2(GalNAca1-3)Galb1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'masked_sequence': 'Fuca1-2(GalNAca1-3)<mask>b1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'expected_token': 'Gal',
        'target_token_type': 'residue',
    },
    'glyberta': {
        'base_sequence': 'Fuca1-2(GalNAca1-3)Galb1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'masked_sequence': 'Fuca1-2(GalNAca1-3)<mask>b1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'expected_token': 'Gal',
        'target_token_type': 'residue',
    },
    'hybrid_char_bpe': {
        'base_sequence': 'Fuca1-2(GalNAca1-3)Galb1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'masked_sequence': 'Fuca1-2(GalNAca1-3)<mask>b1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'expected_token': 'Gal',
        'target_token_type': 'merged_residue',
    },
    'byte_bpe': {
        'base_sequence': 'Fuca1-2(GalNAca1-3)Galb1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'masked_sequence': 'Fuca1-2(GalNAca1-3)<mask>b1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'expected_token': 'Gal',
        'target_token_type': 'byte_bpe_residue',
    },
    'linkage_block': {
        'base_sequence': 'Fuca1-2(GalNAca1-3)Galb1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'masked_sequence': 'Fuca1-2(GalNAca1-3)<mask>(Galb1-4GlcNAcb1-6)GalNAca',
        'expected_token': 'Galb1-3',
        'target_token_type': 'linkage_block',
    },
    'donor_bound': {
        'base_sequence': 'Fuca1-2(GalNAca1-3)Galb1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'masked_sequence': 'Fuca1-2(GalNAca1-3)<mask>-3(Galb1-4GlcNAcb1-6)GalNAca',
        'expected_token': 'Galb1',
        'target_token_type': 'donor_bound',
    },
    'semi_atomic': {
        'base_sequence': 'Fuca1-2(GalNAca1-3)Galb1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'masked_sequence': 'Fuca1-2(GalNAca1-3)<mask>b1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'expected_token': 'Gal',
        'target_token_type': 'residue',
    },
}

family_key = TOKENIZER_FAMILY.lower().replace('-', '_')
if family_key not in example_probe_by_family:
    raise ValueError(f'No example probe is configured for tokenizer family: {TOKENIZER_FAMILY}')

example_probe = example_probe_by_family[family_key]
encoded_example = tokenizer(example_probe['masked_sequence'], return_tensors='pt')
input_ids = encoded_example['input_ids'].to(device)
attention_mask = encoded_example.get('attention_mask')
if attention_mask is not None:
    attention_mask = attention_mask.to(device)

mask_positions = (input_ids[0] == tokenizer.mask_token_id).nonzero(as_tuple=False).flatten()
if len(mask_positions) != 1:
    raise ValueError(
        'Expected exactly one mask token in the example probe, '
        f'but found {len(mask_positions)} positions.'
    )

with torch.no_grad():
    example_outputs = model(input_ids=input_ids, attention_mask=attention_mask)

mask_index = int(mask_positions.item())
mask_probs = torch.softmax(example_outputs.logits[0, mask_index], dim=-1)
top_scores, top_token_ids = torch.topk(mask_probs, k=3)

display(
    pd.DataFrame(
        {
            'field': ['tokenizer_family', 'base_sequence', 'masked_sequence', 'expected_token', 'target_token_type'],
            'value': [
                TOKENIZER_FAMILY,
                example_probe['base_sequence'],
                example_probe['masked_sequence'],
                example_probe['expected_token'],
                example_probe['target_token_type'],
            ],
        }
    )
)

example_top3_rows = []
for rank, (token_id, score) in enumerate(zip(top_token_ids.tolist(), top_scores.tolist()), start=1):
    raw_token = tokenizer.convert_ids_to_tokens([int(token_id)])[0]
    decoded_token = tokenizer.decode([int(token_id)], skip_special_tokens=True).strip()
    display_token = decoded_token if decoded_token else raw_token.strip()
    example_top3_rows.append(
        {
            'rank': rank,
            'predicted_token': display_token,
            'raw_token': raw_token,
            'probability': float(score),
            'matches_expected': display_token == example_probe['expected_token'],
        }
    )

display(pd.DataFrame(example_top3_rows))
